# 🏍️ 오토바이 야간 주행 시맨틱 세그멘테이션 + HUD 시뮬레이션

## AIFFEL DLThon 팀프로젝트

**프로젝트 구성:**
- Part 1: 데이터 준비 (Kaggle 다운로드 → COCO JSON → 마스크)
- Part 2: U-Net 모델 학습 (Weighted CE vs Plain CE 비교)
- Part 3: 평가 및 시각화 (IoU, Confusion Matrix)
- Part 4: 비전 데모 - JARVIS 스타일 HUD 시뮬레이션
- Part 5: 생체 신호 통합 - 건강 경고 시스템

**실행 환경:**
- Google Colab (GPU T4 이상 권장)
- Python 3.10+
- PyTorch 2.0+

**데이터셋:** Motorcycle Night Ride Semantic Segmentation (Kaggle)

---


# 📦 Part 1: 환경 설정 및 데이터 다운로드

## 1-1. Kaggle API 세팅

**사전 준비:**
1. [kaggle.com](https://kaggle.com) → Settings → API → Create New Token
2. 다운받은 `kaggle.json` 파일을 노트북 환경에 업로드


In [ ]:
# 필수 라이브러리 설치
!pip install kaggle pycocotools albumentations imageio imageio-ffmpeg -q

import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

# kaggle.json 업로드 후 아래 실행
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✅ Kaggle API 세팅 완료")

## 1-2. 데이터 다운로드

데이터셋: `sadhliroomyprime/motorcycle-night-ride-semantic-segmentation`  
크기: 약 340MB (200 프레임, 1920x1080)


In [ ]:
# 데이터 다운로드
!kaggle datasets download -d sadhliroomyprime/motorcycle-night-ride-semantic-segmentation

# 압축 풀기
!unzip -q motorcycle-night-ride-semantic-segmentation.zip -d ./motorcycle_data

# 긴 폴더명을 짧은 심볼릭 링크로
!ln -sf "./motorcycle_data/www.acmeai.tech ODataset 1 - Motorcycle Night Ride Dataset" ./data

print("✅ 데이터 다운로드 완료")
!ls ./data

# 🔍 Part 2: 데이터 탐색 및 마스크 생성

## 2-1. COCO JSON 구조 확인


In [ ]:
import json
import os

DATA_ROOT = './data'
IMAGES_DIR = os.path.join(DATA_ROOT, 'images')
JSON_PATH = os.path.join(DATA_ROOT, 'COCO_motorcycle (pixel).json')

print("📖 COCO JSON 읽는 중...")
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    coco = json.load(f)
print("✅ 완료!\n")

print(f"📋 최상위 키: {list(coco.keys())}")
print(f"\n🏷️ 클래스 정보:")
for cat in coco.get('categories', []):
    print(f"  id={cat['id']:8d}  name={cat['name']}")

print(f"\n📷 총 이미지 수: {len(coco['images'])}")
print(f"🎨 총 어노테이션 수: {len(coco['annotations'])}")

## 2-2. COCO → 마스크 변환 및 시각화

COCO ID(7자리) → 우리가 쓸 0~5로 매핑


In [ ]:
import numpy as np
from pycocotools.coco import COCO
import matplotlib.pyplot as plt
from PIL import Image

coco_api = COCO(JSON_PATH)

# Category ID → 0~5 매핑
CATEGORY_MAPPING = {
    1323880: 0,  # Undrivable
    1323881: 1,  # Road
    1323882: 2,  # Lane Mark
    1323885: 3,  # My bike
    1329681: 4,  # Rider
    1323884: 5,  # Moveable
}

CLASS_NAMES = ['Undrivable', 'Road', 'Lane Mark', 'My bike', 'Rider', 'Moveable']

COLORS = np.array([
    [128, 128, 128],  # 0: Undrivable
    [ 64,  64,  64],  # 1: Road
    [255, 255,   0],  # 2: Lane Mark
    [  0,   0, 255],  # 3: My bike
    [255,   0,   0],  # 4: Rider
    [  0, 255,   0],  # 5: Moveable
], dtype=np.uint8)


def get_mask_for_image(image_id):
    """특정 이미지의 모든 어노테이션을 하나의 마스크로 합침"""
    img_info = coco_api.loadImgs(image_id)[0]
    H, W = img_info['height'], img_info['width']
    mask = np.full((H, W), 255, dtype=np.uint8)
    
    ann_ids = coco_api.getAnnIds(imgIds=image_id)
    anns = coco_api.loadAnns(ann_ids)
    anns_sorted = sorted(anns, key=lambda x: x['area'], reverse=True)
    
    for ann in anns_sorted:
        cat_id = ann['category_id']
        if cat_id not in CATEGORY_MAPPING:
            continue
        our_class = CATEGORY_MAPPING[cat_id]
        binary = coco_api.annToMask(ann)
        mask[binary == 1] = our_class
    
    return mask, img_info['file_name']


# 첫 번째 이미지로 테스트 시각화
first_id = coco_api.getImgIds()[0]
mask, fname = get_mask_for_image(first_id)

img = np.array(Image.open(os.path.join(IMAGES_DIR, fname)).convert('RGB'))
color_mask = np.zeros((*mask.shape, 3), dtype=np.uint8)
for cls in range(6):
    color_mask[mask == cls] = COLORS[cls]
overlay = (img * 0.5 + color_mask * 0.5).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
axes[0].imshow(img);        axes[0].set_title(f'Original: {fname}');  axes[0].axis('off')
axes[1].imshow(color_mask); axes[1].set_title('Generated Mask');      axes[1].axis('off')
axes[2].imshow(overlay);    axes[2].set_title('Overlay');             axes[2].axis('off')
plt.tight_layout()
plt.show()

print(f"✅ 마스크 생성 성공")
print(f"   Shape: {mask.shape}, 클래스: {np.unique(mask)}")

## 2-3. 200장 전체 마스크 생성 및 저장

In [ ]:
from tqdm import tqdm

MASKS_DIR = './data/masks_generated'
os.makedirs(MASKS_DIR, exist_ok=True)

image_ids = coco_api.getImgIds()
print(f"🎨 총 {len(image_ids)}개 마스크 생성 시작...")

success_count = 0
for img_id in tqdm(image_ids, desc="Generating masks"):
    try:
        mask, fname = get_mask_for_image(img_id)
        save_path = os.path.join(MASKS_DIR, fname)
        Image.fromarray(mask).save(save_path)
        success_count += 1
    except Exception as e:
        print(f"실패: img_id={img_id}, {e}")

print(f"\n✅ 성공: {success_count}/{len(image_ids)}")

## 2-4. 클래스 분포 분석 (핵심 발견!)

In [ ]:
from collections import Counter

total_pixels = Counter()
mask_files = sorted(os.listdir(MASKS_DIR))

print("📊 전체 데이터 클래스 분포 계산 중...")
for mf in tqdm(mask_files):
    mask = np.array(Image.open(os.path.join(MASKS_DIR, mf)))
    values, counts = np.unique(mask, return_counts=True)
    for v, c in zip(values, counts):
        total_pixels[int(v)] += int(c)

total = sum(total_pixels.values())
print("\n📊 클래스별 픽셀 비율:")
for cls_id in sorted(total_pixels.keys()):
    ratio = total_pixels[cls_id] / total * 100
    name = CLASS_NAMES[cls_id] if cls_id < 6 else 'Background'
    bar = '█' * int(ratio / 2)
    print(f"  {cls_id:3d} {name:15s} {ratio:5.2f}% {bar}")

# 가중치 계산용 저장
class_ratios = [total_pixels.get(i, 0) / total for i in range(6)]
print(f"\n⚠️ 심각한 불균형 발견: Lane Mark가 1.4% 수준 → Weighted Loss 필요!")

# 🔧 Part 3: PyTorch 환경 및 Dataset

## 3-1. GPU 확인


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"메모리: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
    print("⚠️ CPU 모드 (학습 매우 느림)")

## 3-2. Train/Val 분할 (80:20)

In [ ]:
# 유효한 파일 쌍 확인
all_filenames = [img_info['file_name'] for img_info in coco['images']]
valid_filenames = []
for fn in all_filenames:
    if os.path.exists(os.path.join(IMAGES_DIR, fn)) and \
       os.path.exists(os.path.join(MASKS_DIR, fn)):
        valid_filenames.append(fn)

print(f"✅ 유효한 쌍: {len(valid_filenames)}개")

# 재현성 확보를 위한 seed 고정 분할
np.random.seed(42)
shuffled = np.random.permutation(valid_filenames)

split_idx = int(len(shuffled) * 0.8)
train_files = shuffled[:split_idx].tolist()
val_files   = shuffled[split_idx:].tolist()

print(f"🏋️ 학습용: {len(train_files)}개")
print(f"🧪 검증용: {len(val_files)}개")

## 3-3. Dataset 클래스 및 DataLoader

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 학습용 증강 (HFlip + Brightness/Contrast)
train_transform = A.Compose([
    A.Resize(height=256, width=512),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# 검증용 (증강 없음)
val_transform = A.Compose([
    A.Resize(height=256, width=512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


class MotorcycleDataset(Dataset):
    def __init__(self, file_list, images_dir, masks_dir, transform=None):
        self.file_list  = file_list
        self.images_dir = images_dir
        self.masks_dir  = masks_dir
        self.transform  = transform
    
    def __len__(self):
        return len(self.file_list)
    
    def __getitem__(self, idx):
        fname = self.file_list[idx]
        image = np.array(Image.open(os.path.join(self.images_dir, fname)).convert('RGB'))
        mask = np.array(Image.open(os.path.join(self.masks_dir, fname)))
        mask[mask == 255] = 0  # 배경을 Undrivable로 통합
        
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug['image']
            mask  = aug['mask'].long()
        
        return image, mask


train_dataset = MotorcycleDataset(train_files, IMAGES_DIR, MASKS_DIR, train_transform)
val_dataset   = MotorcycleDataset(val_files,   IMAGES_DIR, MASKS_DIR, val_transform)

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"✅ Train: {len(train_dataset)}개, Val: {len(val_dataset)}개")
print(f"✅ Batch size: {BATCH_SIZE}")

# 🏗️ Part 4: U-Net 모델 구현

## 4-1. U-Net 구성 요소

세그멘테이션의 표준 아키텍처. 인코더(의미 파악) + 디코더(픽셀 색칠) + 스킵 커넥션(세부 정보 전달).


In [ ]:
import torch.nn.functional as F


class DoubleConv(nn.Module):
    """(Conv3x3 → BN → ReLU) × 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """MaxPool → DoubleConv (인코더)"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool_conv = nn.Sequential(
            nn.MaxPool2d(2, 2),
            DoubleConv(in_channels, out_channels)
        )
    
    def forward(self, x):
        return self.pool_conv(x)


class Up(nn.Module):
    """Upsample + skip connection + DoubleConv (디코더)"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, 2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x1, x2):
        x1 = self.up(x1)
        diff_h = x2.size(2) - x1.size(2)
        diff_w = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diff_w // 2, diff_w - diff_w // 2,
                         diff_h // 2, diff_h - diff_h // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, num_classes, kernel_size=1)
    
    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=6):
        super().__init__()
        self.inc   = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)
        self.up1   = Up(1024, 512)
        self.up2   = Up(512, 256)
        self.up3   = Up(256, 128)
        self.up4   = Up(128, 64)
        self.outc  = OutConv(64, num_classes)
    
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x,  x3)
        x = self.up3(x,  x2)
        x = self.up4(x,  x1)
        return self.outc(x)


# 모델 생성 & 테스트
model = UNet(in_channels=3, num_classes=6).to(device)

dummy = torch.randn(2, 3, 256, 512).to(device)
output = model(dummy)

print(f"📥 입력 shape:  {dummy.shape}")
print(f"📤 출력 shape:  {output.shape}")
print(f"📊 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

# 🎯 Part 5: 학습 설정

## 5-1. Weighted Loss + Optimizer + Metric


In [ ]:
# ===== Weighted Cross Entropy Loss =====
class_ratios_t = torch.tensor(class_ratios)
class_weights = 1.0 / (class_ratios_t + 1e-6)
class_weights = class_weights / class_weights.mean() * 6
class_weights = class_weights.to(device)

print("📊 클래스별 가중치 (Weighted CE):")
for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights)):
    print(f"  {i} {name:12s} 가중치: {w.item():.2f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)


# ===== Optimizer =====
LEARNING_RATE = 1e-4

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)


# ===== IoU 계산 =====
def compute_iou(pred, target, num_classes=6):
    ious = []
    pred   = pred.cpu().numpy()
    target = target.cpu().numpy()
    
    for cls in range(num_classes):
        pred_cls   = (pred == cls)
        target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum()
        union        = (pred_cls | target_cls).sum()
        iou = float('nan') if union == 0 else intersection / union
        ious.append(iou)
    return ious


print("\n✅ Loss / Optimizer / IoU 준비 완료")

## 5-2. 학습/검증 함수

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    pbar = tqdm(loader, desc='Training')
    for imgs, masks in pbar:
        imgs  = imgs.to(device)
        masks = masks.to(device)
        
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / num_batches


def validate(model, loader, criterion, device, num_classes=6):
    model.eval()
    total_loss = 0.0
    num_batches = 0
    all_ious = [[] for _ in range(num_classes)]
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validating')
        for imgs, masks in pbar:
            imgs  = imgs.to(device)
            masks = masks.to(device)
            
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            
            preds = outputs.argmax(dim=1)
            batch_ious = compute_iou(preds, masks, num_classes)
            for cls, iou in enumerate(batch_ious):
                if not np.isnan(iou):
                    all_ious[cls].append(iou)
            
            total_loss += loss.item()
            num_batches += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    class_ious = [np.mean(ious) if ious else 0.0 for ious in all_ious]
    mIoU = np.mean(class_ious)
    return total_loss / num_batches, class_ious, mIoU

# 🚀 Part 6: 학습 실행 (실험 1: Weighted CE)

30 에포크, T4 GPU 기준 약 15~20분 소요


In [ ]:
NUM_EPOCHS = 30
BEST_IOU = 0.0
SAVE_PATH = './best_model.pth'

history = {'train_loss': [], 'val_loss': [], 'val_mIoU': [], 'class_iou': []}

print(f"🚀 학습 시작! (Weighted CE, {NUM_EPOCHS} 에포크)\n")

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n{'='*60}\n📍 Epoch {epoch}/{NUM_EPOCHS}\n{'='*60}")
    
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, class_ious, mIoU = validate(model, val_loader, criterion, device)
    scheduler.step(mIoU)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_mIoU'].append(mIoU)
    history['class_iou'].append(class_ious)
    
    print(f"\n📊 Train: {train_loss:.4f} | Val: {val_loss:.4f} | mIoU: {mIoU:.4f}")
    for i, (name, iou) in enumerate(zip(CLASS_NAMES, class_ious)):
        print(f"   {i} {name:12s} {iou:.4f}")
    
    if mIoU > BEST_IOU:
        BEST_IOU = mIoU
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"   💾 최고 성능 갱신!")

print(f"\n\n🎉 학습 완료! 최고 mIoU: {BEST_IOU:.4f}")

# 🧪 Part 7: 비교 실험 (실험 2: Plain CE)

같은 조건에서 가중치 없는 CE Loss로 학습 → Weighted CE의 효과 검증


In [ ]:
# 새 모델 생성 (초기 가중치로)
model_plain = UNet(in_channels=3, num_classes=6).to(device)
criterion_plain = nn.CrossEntropyLoss()  # ⭐ 가중치 없음

optimizer_plain = optim.Adam(model_plain.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler_plain = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_plain, mode='max', factor=0.5, patience=5
)

BEST_IOU_PLAIN = 0.0
SAVE_PATH_PLAIN = './best_model_plain.pth'
history_plain = {'train_loss': [], 'val_loss': [], 'val_mIoU': [], 'class_iou': []}

print(f"🚀 비교 실험 시작: Plain CE ({NUM_EPOCHS} 에포크)\n")

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n{'='*60}\n📍 [Plain CE] Epoch {epoch}/{NUM_EPOCHS}\n{'='*60}")
    
    train_loss = train_one_epoch(model_plain, train_loader, criterion_plain, optimizer_plain, device)
    val_loss, class_ious, mIoU = validate(model_plain, val_loader, criterion_plain, device)
    scheduler_plain.step(mIoU)
    
    history_plain['train_loss'].append(train_loss)
    history_plain['val_loss'].append(val_loss)
    history_plain['val_mIoU'].append(mIoU)
    history_plain['class_iou'].append(class_ious)
    
    print(f"\n📊 Train: {train_loss:.4f} | Val: {val_loss:.4f} | mIoU: {mIoU:.4f}")
    for i, (name, iou) in enumerate(zip(CLASS_NAMES, class_ious)):
        print(f"   {i} {name:12s} {iou:.4f}")
    
    if mIoU > BEST_IOU_PLAIN:
        BEST_IOU_PLAIN = mIoU
        torch.save(model_plain.state_dict(), SAVE_PATH_PLAIN)

print(f"\n\n🎉 Plain CE 학습 완료! 최고 mIoU: {BEST_IOU_PLAIN:.4f}")

# 📊 Part 8: 결과 분석 및 시각화

## 8-1. 학습 곡선


In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].plot(epochs_range, history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss',   marker='s')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training / Validation Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['val_mIoU'], marker='o', color='green')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU')
axes[1].set_title('Validation mIoU')
axes[1].grid(True, alpha=0.3)

class_iou_history = np.array(history['class_iou'])
for cls_idx in range(6):
    axes[2].plot(epochs_range, class_iou_history[:, cls_idx],
                 label=CLASS_NAMES[cls_idx], marker='.')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('IoU per Class')
axes[2].set_title('Per-class IoU over Epochs')
axes[2].legend(loc='lower right', fontsize=8); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

## 8-2. Weighted vs Plain 비교

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# mIoU 비교
axes[0, 0].plot(epochs_range, history['val_mIoU'],       label='Weighted CE', marker='o', color='green', linewidth=2)
axes[0, 0].plot(epochs_range, history_plain['val_mIoU'], label='Plain CE',    marker='s', color='orange', linewidth=2)
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('mIoU')
axes[0, 0].set_title('mIoU Comparison')
axes[0, 0].legend(fontsize=11); axes[0, 0].grid(True, alpha=0.3)

# Val Loss 비교
axes[0, 1].plot(epochs_range, history['val_loss'],       label='Weighted CE', marker='o', color='green', linewidth=2)
axes[0, 1].plot(epochs_range, history_plain['val_loss'], label='Plain CE',    marker='s', color='orange', linewidth=2)
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Val Loss')
axes[0, 1].set_title('Validation Loss Comparison')
axes[0, 1].legend(fontsize=11); axes[0, 1].grid(True, alpha=0.3)

# 클래스별 IoU 막대 비교
x = np.arange(6)
width = 0.35
final_weighted = history['class_iou'][-1]
final_plain    = history_plain['class_iou'][-1]

bars1 = axes[1, 0].bar(x - width/2, final_weighted, width, label='Weighted CE', color='green', alpha=0.8)
bars2 = axes[1, 0].bar(x + width/2, final_plain,    width, label='Plain CE',    color='orange', alpha=0.8)

for bars in [bars1, bars2]:
    for bar in bars:
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                        f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

axes[1, 0].set_xticks(x); axes[1, 0].set_xticklabels(CLASS_NAMES, rotation=30)
axes[1, 0].set_ylabel('Final IoU')
axes[1, 0].set_title('Final Per-class IoU')
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3, axis='y')
axes[1, 0].set_ylim(0, 1.0)

# 개선률
improvements = [(w - p) / (p + 1e-6) * 100 for w, p in zip(final_weighted, final_plain)]
colors = ['green' if imp > 0 else 'red' for imp in improvements]
bars = axes[1, 1].barh(CLASS_NAMES, improvements, color=colors, alpha=0.7)
axes[1, 1].set_xlabel('Improvement (%)')
axes[1, 1].set_title('Relative Improvement: Weighted vs Plain')
axes[1, 1].axvline(x=0, color='black', linewidth=0.5)
axes[1, 1].grid(True, alpha=0.3, axis='x')

for bar, imp in zip(bars, improvements):
    axes[1, 1].text(imp + (2 if imp > 0 else -2), bar.get_y() + bar.get_height()/2,
                    f'{imp:+.1f}%', va='center', ha='left' if imp > 0 else 'right')

plt.tight_layout()
plt.savefig('comparison.png', dpi=100, bbox_inches='tight')
plt.show()

# 표
print("\n" + "="*70)
print("📊 최종 결과 비교")
print("="*70)
print(f"{'Class':<15s} | {'Weighted':>10s} | {'Plain':>10s} | {'Diff':>10s} | {'Rel.%':>8s}")
print("-" * 70)
for i, name in enumerate(CLASS_NAMES):
    w, p = final_weighted[i], final_plain[i]
    print(f"{name:<15s} | {w:>10.4f} | {p:>10.4f} | {w-p:>+10.4f} | {(w-p)/(p+1e-6)*100:>+7.1f}%")
print("-" * 70)
print(f"{'mIoU':<15s} | {np.mean(final_weighted):>10.4f} | {np.mean(final_plain):>10.4f}")
print("="*70)

## 8-3. Confusion Matrix (Weighted 모델)

In [ ]:
# 최고 모델 로드
model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

confusion = np.zeros((6, 6), dtype=np.int64)

with torch.no_grad():
    for imgs, masks in val_loader:
        imgs  = imgs.to(device)
        masks = masks.numpy()
        outputs = model(imgs)
        preds   = outputs.argmax(dim=1).cpu().numpy()
        
        for t in range(6):
            for p in range(6):
                confusion[t, p] += ((masks == t) & (preds == p)).sum()

confusion_norm = confusion / (confusion.sum(axis=1, keepdims=True) + 1e-10)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(confusion_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(6)); ax.set_yticks(range(6))
ax.set_xticklabels(CLASS_NAMES, rotation=45)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('Ground Truth')
ax.set_title('Confusion Matrix (row-normalized)')

for i in range(6):
    for j in range(6):
        v = confusion_norm[i, j]
        color = 'white' if v > 0.5 else 'black'
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', color=color)

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

## 8-4. 예측 시각화 (6개 샘플)

In [ ]:
def denormalize(img_tensor):
    """정규화 복원"""
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img  = img * std + mean
    return np.clip(img * 255, 0, 255).astype(np.uint8)


def mask_to_color(mask):
    color = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls in range(6):
        color[mask == cls] = COLORS[cls]
    return color


model.eval()
fig, axes = plt.subplots(6, 3, figsize=(18, 24))

count = 0
with torch.no_grad():
    for imgs, masks in val_loader:
        if count >= 6: break
        outputs = model(imgs.to(device))
        preds   = outputs.argmax(dim=1).cpu().numpy()
        
        for i in range(imgs.shape[0]):
            if count >= 6: break
            orig   = denormalize(imgs[i])
            gt     = mask_to_color(masks[i].numpy())
            pred_c = mask_to_color(preds[i])
            
            axes[count, 0].imshow(orig);   axes[count, 0].set_title(f'Sample {count+1}: Original'); axes[count, 0].axis('off')
            axes[count, 1].imshow(gt);     axes[count, 1].set_title('Ground Truth'); axes[count, 1].axis('off')
            axes[count, 2].imshow(pred_c); axes[count, 2].set_title('Prediction');   axes[count, 2].axis('off')
            count += 1

plt.tight_layout()
plt.savefig('predictions.png', dpi=80, bbox_inches='tight')
plt.show()

# 🎨 Part 9: 비전 확장 - HUD 데모 시뮬레이션

"저희가 그리는 미래: 라이더 시야 확장 시스템"  
세그멘테이션 결과 → 위험도 분석 → HUD 시각화

## 9-1. 위험도 분석 엔진 (룰 기반)


In [ ]:
def analyze_danger(mask, img_shape=(256, 512)):
    """
    세그멘테이션 마스크 → 위험도 분석
    3x3 영역별 위험 점수 + 경고 메시지 생성
    """
    H, W = mask.shape
    third_h, third_w = H // 3, W // 3
    
    regions = {
        'top_left':    mask[:third_h,        :third_w],
        'top_center':  mask[:third_h,        third_w:2*third_w],
        'top_right':   mask[:third_h,        2*third_w:],
        'mid_left':    mask[third_h:2*third_h, :third_w],
        'mid_center':  mask[third_h:2*third_h, third_w:2*third_w],
        'mid_right':   mask[third_h:2*third_h, 2*third_w:],
        'bot_left':    mask[2*third_h:,      :third_w],
        'bot_center':  mask[2*third_h:,      third_w:2*third_w],
        'bot_right':   mask[2*third_h:,      2*third_w:],
    }
    
    danger_info = {
        'overall_score': 0,
        'level': 'SAFE',
        'warnings': [],
        'direction': None,
        'region_scores': {},
    }
    
    # 전방 움직이는 물체
    front_mv = (regions['mid_center'] == 5).mean()
    if front_mv > 0.10:
        score = min(100, front_mv * 300)
        danger_info['overall_score'] += score
        danger_info['warnings'].append({'text': 'FRONT OBJECT', 'severity': min(3, int(score / 30)), 'position': 'center'})
    
    # 측면 위험
    if (regions['mid_left'] == 5).mean() > 0.08:
        danger_info['overall_score'] += 30
        danger_info['warnings'].append({'text': 'LEFT HAZARD', 'severity': 2, 'position': 'left'})
    if (regions['mid_right'] == 5).mean() > 0.08:
        danger_info['overall_score'] += 30
        danger_info['warnings'].append({'text': 'RIGHT HAZARD', 'severity': 2, 'position': 'right'})
    
    # 차선 이탈
    bike_cols = np.where((mask == 3).any(axis=0))[0]
    lane_cols = np.where((mask == 2).any(axis=0))[0]
    if len(bike_cols) > 0 and len(lane_cols) > 0:
        min_dist = np.min(np.abs(lane_cols - bike_cols.mean()))
        if min_dist < W * 0.05:
            danger_info['overall_score'] += 25
            danger_info['warnings'].append({'text': 'LANE DEVIATION', 'severity': 2, 'position': 'center'})
    
    # 도로 이탈
    if (regions['mid_center'] == 0).mean() > 0.40:
        danger_info['overall_score'] += 20
        danger_info['warnings'].append({'text': 'LIMITED ROAD', 'severity': 1, 'position': 'center'})
    
    # 영역별 점수
    for rname, rmask in regions.items():
        mv = (rmask == 5).mean()
        un = (rmask == 0).mean()
        danger_info['region_scores'][rname] = min(100, (mv * 70 + un * 30) * 100)
    
    # 레벨
    danger_info['overall_score'] = min(100, danger_info['overall_score'])
    if danger_info['overall_score'] >= 70:   danger_info['level'] = 'DANGER'
    elif danger_info['overall_score'] >= 40: danger_info['level'] = 'WARNING'
    elif danger_info['overall_score'] >= 15: danger_info['level'] = 'CAUTION'
    else:                                    danger_info['level'] = 'SAFE'
    
    return danger_info


print("✅ 위험도 분석 함수 준비 완료")

## 9-2. 생체 신호 시뮬레이터

실제 시스템에선 PPG/EEG 센서에서 받지만, 데모에선 시나리오 기반 시뮬레이션


In [ ]:
class BiometricScenario:
    """
    시나리오 기반 생체 시뮬레이터
    Phase 1: 평온 → Phase 2: 위험 → Phase 3: 건강 경고
    """
    def __init__(self, total_frames=90, seed=42):
        np.random.seed(seed)
        self.total_frames = total_frames
        self.phase1_end = int(total_frames * 0.25)
        self.phase2_end = int(total_frames * 0.55)
        
        self.hr = 72.0
        self.fatigue = 10.0
        self.activity = 45.0
        self.focus = 90.0
        self.stress = 15.0
        
        self.health_alert = False
        self.health_alert_start = None
    
    def update(self, frame_idx, danger_level):
        # Phase 1: 평온
        if frame_idx < self.phase1_end:
            target_hr = 72 + np.random.randn() * 2
            fat_rate = 0.1
            target_act = 45
            target_focus = 90
        # Phase 2: 위험
        elif frame_idx < self.phase2_end:
            progress = (frame_idx - self.phase1_end) / (self.phase2_end - self.phase1_end)
            target_hr = 72 + progress * 70 + np.random.randn() * 3
            fat_rate = 1.2
            target_act = 70 + progress * 25
            target_focus = 90 - progress * 40
        # Phase 3: 건강 경고
        else:
            target_hr = 145 + np.random.randn() * 5
            fat_rate = 1.8
            target_act = 85
            target_focus = 35
            if not self.health_alert and self.fatigue > 65:
                self.health_alert = True
                self.health_alert_start = frame_idx
        
        self.hr += (target_hr - self.hr) * 0.25
        self.fatigue += fat_rate
        self.activity += (target_act - self.activity) * 0.3
        self.focus += (target_focus - self.focus) * 0.2
        
        self.hr = np.clip(self.hr, 55, 180)
        self.fatigue = np.clip(self.fatigue, 0, 100)
        self.activity = np.clip(self.activity, 0, 100)
        self.focus = np.clip(self.focus, 0, 100)
        self.stress = np.clip((self.hr - 60) * 0.7 + self.fatigue * 0.2, 0, 100)
        
        return {
            'hr': self.hr, 'fatigue': self.fatigue, 'activity': self.activity,
            'stress': self.stress, 'focus': self.focus,
            'health_alert': self.health_alert,
            'health_alert_duration': (frame_idx - self.health_alert_start) if self.health_alert_start else 0,
        }


def _get_hr_zone(hr):
    if hr < 85:   return 'NORMAL',   (80, 255, 150)
    elif hr < 105:return 'ELEVATED', (255, 220, 50)
    elif hr < 125:return 'HIGH',     (255, 140, 30)
    elif hr < 145:return 'CRITICAL', (255, 100, 60)
    else:         return 'EXTREME',  (255, 60, 80)


print("✅ 생체 시뮬레이터 준비 완료")

## 9-3. JARVIS 스타일 HUD 렌더링

**디자인 철학:**
- 시야 중앙 절대 가리지 않음
- 정보는 4개 모서리에만
- 반투명 + 얇은 네온 라인
- 위험 시 가장자리만 빛남


In [ ]:
import cv2


def render_hud(image, danger_info, bio_data, mask=None, frame_idx=0):
    """JARVIS 스타일 미니멀 HUD + 건강 모니터링"""
    H, W = image.shape[:2]
    canvas = image.copy()
    
    CYAN   = (0, 230, 255)
    GREEN  = (80, 255, 150)
    YELLOW = (255, 220, 50)
    ORANGE = (255, 140, 30)
    RED    = (255, 60, 80)
    PINK   = (255, 100, 140)
    WHITE  = (240, 245, 255)
    DIM    = (140, 160, 180)
    
    level_color = {
        'SAFE': GREEN, 'CAUTION': YELLOW,
        'WARNING': ORANGE, 'DANGER': RED,
    }.get(danger_info['level'], CYAN)
    
    is_health_alert = bio_data.get('health_alert', False)
    
    # 좌상단: 속도
    cv2.putText(canvas, '62', (20, 30), cv2.FONT_HERSHEY_DUPLEX, 0.6, WHITE, 1)
    cv2.putText(canvas, 'km/h', (50, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.35, DIM, 1)
    
    # 우상단: 시간 + 집중도
    cv2.putText(canvas, '14:23', (W - 70, 22), cv2.FONT_HERSHEY_DUPLEX, 0.5, WHITE, 1)
    focus_val = bio_data['focus']
    focus_color = GREEN if focus_val > 70 else (YELLOW if focus_val > 50 else RED)
    cv2.putText(canvas, f"FOCUS {focus_val:.0f}%", (W - 85, 38),
                cv2.FONT_HERSHEY_SIMPLEX, 0.32, focus_color, 1)
    
    # 좌하단: 미니맵
    mini_x, mini_y = 20, H - 45
    mini_size = 28
    cell = mini_size // 3
    region_order = [
        ['top_left', 'top_center', 'top_right'],
        ['mid_left', 'mid_center', 'mid_right'],
        ['bot_left', 'bot_center', 'bot_right'],
    ]
    for row_idx, row in enumerate(region_order):
        for col_idx, rname in enumerate(row):
            score = danger_info['region_scores'].get(rname, 0)
            if score > 50:   color = RED
            elif score > 25: color = ORANGE
            elif score > 10: color = YELLOW
            else:            color = (50, 60, 75)
            
            x1 = mini_x + col_idx * cell
            y1 = mini_y + row_idx * cell
            overlay = canvas.copy()
            cv2.rectangle(overlay, (x1, y1), (x1 + cell - 1, y1 + cell - 1), color, -1)
            canvas[:] = cv2.addWeighted(overlay, 0.65, canvas, 0.35, 0)
    
    # 생체 정보 3종
    bio_x_start = mini_x + mini_size + 18
    bio_y = H - 25
    bio_label_y = H - 38
    
    # HR
    hr_val = bio_data['hr']
    _, hr_color = _get_hr_zone(hr_val)
    heart_pulse = (np.sin(frame_idx * (hr_val / 60) * 0.5) + 1) / 2
    heart_size = int(3 + heart_pulse * 2)
    cv2.circle(canvas, (bio_x_start + 5, bio_y - 3), heart_size, PINK, -1)
    if hr_val > 120:
        pulse = (np.sin(frame_idx * 0.8) + 1) / 2
        hr_display_color = (255, int(60 + pulse * 100), int(80 + pulse * 100))
    else:
        hr_display_color = hr_color
    cv2.putText(canvas, 'HR', (bio_x_start, bio_label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.3, DIM, 1)
    cv2.putText(canvas, f"{hr_val:.0f}", (bio_x_start + 14, bio_y),
                cv2.FONT_HERSHEY_DUPLEX, 0.5, hr_display_color, 1)
    
    # FAT
    fat_x = bio_x_start + 55
    fat_val = bio_data['fatigue']
    fat_color = GREEN if fat_val < 35 else (YELLOW if fat_val < 55 else (ORANGE if fat_val < 75 else RED))
    cv2.putText(canvas, 'FAT', (fat_x, bio_label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.3, DIM, 1)
    cv2.putText(canvas, f"{fat_val:.0f}%", (fat_x, bio_y),
                cv2.FONT_HERSHEY_DUPLEX, 0.5, fat_color, 1)
    bar_w = 35
    filled = int(bar_w * fat_val / 100)
    cv2.line(canvas, (fat_x, bio_y + 4), (fat_x + bar_w, bio_y + 4), (50, 60, 75), 1)
    if filled > 0:
        cv2.line(canvas, (fat_x, bio_y + 4), (fat_x + filled, bio_y + 4), fat_color, 2)
    
    # ACT
    act_x = fat_x + 55
    act_val = bio_data['activity']
    act_color = CYAN if act_val < 70 else YELLOW
    cv2.putText(canvas, 'ACT', (act_x, bio_label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.3, DIM, 1)
    cv2.putText(canvas, f"{act_val:.0f}%", (act_x, bio_y),
                cv2.FONT_HERSHEY_DUPLEX, 0.5, act_color, 1)
    filled = int(bar_w * act_val / 100)
    cv2.line(canvas, (act_x, bio_y + 4), (act_x + bar_w, bio_y + 4), (50, 60, 75), 1)
    if filled > 0:
        cv2.line(canvas, (act_x, bio_y + 4), (act_x + filled, bio_y + 4), act_color, 2)
    
    # 우하단: 위험도
    dot_x, dot_y = W - 95, H - 28
    cv2.circle(canvas, (dot_x, dot_y), 5, level_color, -1)
    if danger_info['level'] == 'DANGER':
        pulse = (np.sin(frame_idx * 0.8) + 1) / 2
        radius = int(5 + pulse * 5)
        overlay = canvas.copy()
        cv2.circle(overlay, (dot_x, dot_y), radius, level_color, -1)
        canvas[:] = cv2.addWeighted(overlay, 0.4, canvas, 0.6, 0)
    cv2.putText(canvas, f"{danger_info['overall_score']:.0f}",
                (dot_x + 12, dot_y + 5), cv2.FONT_HERSHEY_DUPLEX, 0.5, WHITE, 1)
    cv2.putText(canvas, 'RISK', (dot_x + 38, dot_y + 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.32, DIM, 1)
    
    # 가장자리 위험 표시
    if danger_info['level'] in ['WARNING', 'DANGER']:
        pulse = (np.sin(frame_idx * 0.7) + 1) / 2
        alpha = 0.2 + pulse * 0.25
        glow_width = 25
        overlay = canvas.copy()
        if danger_info['level'] == 'DANGER':
            for i in range(glow_width):
                cv2.line(overlay, (i, 0), (i, H), level_color, 1)
                cv2.line(overlay, (W - 1 - i, 0), (W - 1 - i, H), level_color, 1)
                cv2.line(overlay, (0, i), (W, i), level_color, 1)
                cv2.line(overlay, (0, H - 1 - i), (W, H - 1 - i), level_color, 1)
        else:
            has_left  = any(w['position'] == 'left'   for w in danger_info['warnings'])
            has_right = any(w['position'] == 'right'  for w in danger_info['warnings'])
            has_front = any(w['position'] == 'center' for w in danger_info['warnings'])
            if has_left:
                for i in range(glow_width):
                    cv2.line(overlay, (i, 0), (i, H), level_color, 1)
            if has_right:
                for i in range(glow_width):
                    cv2.line(overlay, (W - 1 - i, 0), (W - 1 - i, H), level_color, 1)
            if has_front:
                for i in range(glow_width):
                    cv2.line(overlay, (0, i), (W, i), level_color, 1)
        canvas[:] = cv2.addWeighted(overlay, alpha, canvas, 1 - alpha, 0)
    
    # 건강 경고 (Phase 3)
    if is_health_alert:
        health_pulse = (np.sin(frame_idx * 0.5) + 1) / 2
        overlay = canvas.copy()
        health_color = (int(255), int(40 + health_pulse * 40), int(100 + health_pulse * 40))
        glow_w = 35
        for i in range(glow_w):
            cv2.line(overlay, (i, 0), (i, H), health_color, 1)
            cv2.line(overlay, (W - 1 - i, 0), (W - 1 - i, H), health_color, 1)
            cv2.line(overlay, (0, i), (W, i), health_color, 1)
            cv2.line(overlay, (0, H - 1 - i), (W, H - 1 - i), health_color, 1)
        canvas[:] = cv2.addWeighted(overlay, 0.3 + health_pulse * 0.15, canvas,
                                     1 - (0.3 + health_pulse * 0.15), 0)
        
        alert_duration = bio_data.get('health_alert_duration', 0)
        if alert_duration < 9:
            banner_h = 100
            banner_y = H // 2 - banner_h // 2
            overlay = canvas.copy()
            cv2.rectangle(overlay, (0, banner_y), (W, banner_y + banner_h), (10, 15, 25), -1)
            canvas[:] = cv2.addWeighted(overlay, 0.85, canvas, 0.15, 0)
            cv2.line(canvas, (0, banner_y), (W, banner_y), health_color, 2)
            cv2.line(canvas, (0, banner_y + banner_h), (W, banner_y + banner_h), health_color, 2)
            
            title = "! HEALTH ALERT !"
            ts = cv2.getTextSize(title, cv2.FONT_HERSHEY_DUPLEX, 0.9, 2)[0]
            cv2.putText(canvas, title, ((W - ts[0]) // 2, banner_y + 35),
                        cv2.FONT_HERSHEY_DUPLEX, 0.9, health_color, 2)
            sub1 = f"HR: {hr_val:.0f} bpm  /  FATIGUE: {fat_val:.0f}%"
            ts1 = cv2.getTextSize(sub1, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)[0]
            cv2.putText(canvas, sub1, ((W - ts1[0]) // 2, banner_y + 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, WHITE, 1)
            sub2 = "PLEASE STOP AND REST"
            ts2 = cv2.getTextSize(sub2, cv2.FONT_HERSHEY_DUPLEX, 0.6, 2)[0]
            cv2.putText(canvas, sub2, ((W - ts2[0]) // 2, banner_y + 85),
                        cv2.FONT_HERSHEY_DUPLEX, 0.6, health_color, 2)
        else:
            banner_h = 28
            overlay = canvas.copy()
            cv2.rectangle(overlay, (0, 48), (W, 48 + banner_h), (20, 10, 20), -1)
            canvas[:] = cv2.addWeighted(overlay, 0.75, canvas, 0.25, 0)
            cv2.line(canvas, (0, 48 + banner_h), (W, 48 + banner_h), health_color, 1)
            icon_x, icon_y = 20, 48 + banner_h // 2
            if int(frame_idx * 0.5) % 2 == 0:
                cv2.circle(canvas, (icon_x, icon_y), 6, health_color, -1)
                cv2.putText(canvas, '!', (icon_x - 2, icon_y + 4),
                            cv2.FONT_HERSHEY_DUPLEX, 0.4, (0, 0, 0), 1)
            msg = f"HEALTH ALERT - REST NOW  (HR: {hr_val:.0f} / FAT: {fat_val:.0f}%)"
            cv2.putText(canvas, msg, (icon_x + 15, icon_y + 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, health_color, 1)
            rec = "FIND SAFE SPOT"
            ts = cv2.getTextSize(rec, cv2.FONT_HERSHEY_DUPLEX, 0.4, 1)[0]
            cv2.putText(canvas, rec, (W - ts[0] - 15, icon_y + 4),
                        cv2.FONT_HERSHEY_DUPLEX, 0.4, WHITE, 1)
    
    return canvas


print("✅ HUD 렌더링 함수 준비 완료")

# 🎬 Part 10: 스토리텔링 데모 비디오

**시나리오:** 평온 주행 → 위험 발생 → 심박 급상승 → 피로 누적 → 🚨 건강 경고 (휴식 권고)

이 영상이 DLThon 발표의 **비전 슬라이드 핵심 자료** 가 됩니다.


In [ ]:
import imageio

print("🎬 스토리텔링 HUD 비디오 생성 중...")
print("   시나리오: 평온 → 위험 → 심박급상승 → 건강경고\n")

total_frames = len(val_dataset)
bio_scenario = BiometricScenario(total_frames=total_frames, seed=42)

# 모든 샘플의 레벨 확인
model.eval()
sample_levels = []
with torch.no_grad():
    for idx in range(total_frames):
        img_tensor, _ = val_dataset[idx]
        pred = model(img_tensor.unsqueeze(0).to(device))
        pred_mask = pred.argmax(dim=1)[0].cpu().numpy()
        danger = analyze_danger(pred_mask)
        sample_levels.append((idx, danger['level']))

safe_samples    = [i for i, l in sample_levels if l == 'SAFE']
caution_samples = [i for i, l in sample_levels if l == 'CAUTION']
warning_samples = [i for i, l in sample_levels if l == 'WARNING']
danger_samples  = [i for i, l in sample_levels if l == 'DANGER']

print(f"  레벨: SAFE={len(safe_samples)}, CAUTION={len(caution_samples)}, "
      f"WARNING={len(warning_samples)}, DANGER={len(danger_samples)}")

# 시나리오 순서
phase1_n = int(total_frames * 0.25)
phase2_n = int(total_frames * 0.30)
phase3_n = total_frames - phase1_n - phase2_n

np.random.seed(42)
safe_pool = (safe_samples + caution_samples) * 3
np.random.shuffle(safe_pool)
phase1_order = safe_pool[:phase1_n]

danger_pool = (warning_samples + danger_samples) * 3 if (warning_samples + danger_samples) else sample_levels * 3
np.random.shuffle(danger_pool)
phase2_order = [x if isinstance(x, int) else x[0] for x in danger_pool[:phase2_n]]

danger_pool2 = danger_samples * 5 if danger_samples else (warning_samples * 5 if warning_samples else [x[0] for x in sample_levels])
np.random.shuffle(danger_pool2)
phase3_order = [x if isinstance(x, int) else x[0] for x in danger_pool2[:phase3_n]]

full_order = list(phase1_order) + list(phase2_order) + list(phase3_order)

# 비디오 생성
frames = []
with torch.no_grad():
    for frame_idx, sample_idx in enumerate(tqdm(full_order, desc="Rendering")):
        img_tensor, _ = val_dataset[sample_idx]
        pred = model(img_tensor.unsqueeze(0).to(device))
        pred_mask = pred.argmax(dim=1)[0].cpu().numpy()
        
        orig_img = denormalize(img_tensor)
        danger = analyze_danger(pred_mask)
        bio_data = bio_scenario.update(frame_idx, danger['level'])
        
        hud_img = render_hud(orig_img, danger, bio_data, mask=pred_mask, frame_idx=frame_idx)
        frames.append(hud_img)

imageio.mimsave('./demo_full.mp4', frames, fps=3, codec='libx264')
print(f"\n✅ demo_full.mp4 생성 완료 ({len(frames)} frames)")

from IPython.display import Video
Video('./demo_full.mp4', embed=True, width=900)

# 🎉 프로젝트 완료!

## 📦 생성된 파일 목록

| 파일 | 설명 |
|------|------|
| `best_model.pth` | Weighted CE로 학습한 최고 성능 모델 |
| `best_model_plain.pth` | Plain CE로 학습한 비교 모델 |
| `training_curves.png` | 학습 곡선 (Loss + mIoU + 클래스별) |
| `comparison.png` | Weighted vs Plain 비교 |
| `confusion_matrix.png` | 혼동 행렬 |
| `predictions.png` | 6개 예측 시각화 |
| `demo_full.mp4` | 🎬 최종 HUD 데모 영상 |

## 🎯 달성한 것

- ✅ 데이터 준비: COCO JSON → 200장 마스크 생성
- ✅ U-Net 구현: 31M 파라미터 베이스라인
- ✅ 비교 실험: Weighted CE vs Plain CE
- ✅ **Lane Mark IoU +102% 개선** (0.156 → 0.316)
- ✅ 비전 확장: JARVIS 스타일 HUD + 생체 신호 모니터링
- ✅ 스토리텔링 데모: 평온 → 위험 → 건강 경고 3단계 시나리오

## 📝 핵심 발견

> **"평균(mIoU)은 0.658 vs 0.657로 거의 같지만, Lane Mark에서는 2배 차이.**  
> **평균 지표에 속지 말고, 안전에 중요한 클래스에 주목해야 한다."**

## 🚀 향후 확장 방향

- Focal Loss / Dice Loss 비교
- ResNet 백본 U-Net
- 객체 탐지(YOLO) + 세그멘테이션 하이브리드
- 실시간 엣지 디바이스 이식 (Jetson)
- 실제 생체 센서(PPG/EEG) 통합
- AR 글래스 / 헬멧 HUD 하드웨어 연동

---

**AIFFEL DLThon 2026**  
_Motorcycle Rider Safety AI Research_
